In [1]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [17]:
# Add helper functions to append conversation
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [ ]:
messages = []
add_user_message(messages, "Write a simple eventbridge json")
chat(messages)

[{'role': 'user', 'content': 'Write a simple eventbridge json'}]


'Here\'s a simple EventBridge rule JSON example:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running", "stopped"]\n  }\n}\n```\n\n---\n\n### What this does:\n- **source** → Listens for events from **EC2**\n- **detail-type** → Filters for **instance state change** events\n- **detail** → Only triggers when state is `running` or `stopped`\n\n---\n\n### Another simple example (Custom Event):\n\n```json\n{\n  "source": ["my.custom.app"],\n  "detail-type": ["OrderPlaced"],\n  "detail": {\n    "status": ["SUCCESS"],\n    "amount": [{"numeric": [">", 100]}]\n  }\n}\n```\n\n---\n\n### Full Rule with Target (CloudWatch/Lambda):\n\n```json\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["my.custom.app"],\n    "detail-type": ["OrderPlaced"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "LambdaTarget",\n      "Arn": "arn:aws:lambda:us-east-1:12

In [18]:
messages = []
add_user_message(messages, "Write a simple eventbridge json")
add_assistant_message(messages, "```json")
print(messages)
chat(messages, stop_sequences=["```"])


[{'role': 'user', 'content': 'Write a simple eventbridge json'}, {'role': 'assistant', 'content': '```json'}]


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011Cb577UWbDGVhq238CBsfY'}

In [ ]:
# Modern replacement solution, for models 4.0 and up
def chat_structured(messages, schema, tool_name="output"):
      response = client.messages.create(
          model=model,
          max_tokens=1000,
          tools=[{
              "name": tool_name,
              "description": "Structured output",
              "input_schema": schema
          }],
          tool_choice={"type": "tool", "name": tool_name},
          messages=messages
      )
      return response.content[0].input 

# Example: force EventBridge JSON structure
schema = {
    "type": "object",
    "properties": {
        "source": {"type": "string"},
        "detail-type": {"type": "string"},
        "detail": {"type": "object"}
    },
    "required": ["source", "detail-type", "detail"]
}

messages = []
add_user_message(messages, "Write a simple EventBridge event")
result = chat_structured(messages, schema)
print(result)  # a Python dict matching the schema

{'source': 'my.app', 'detail-type': 'UserSignedUp', 'detail': {'userId': '12345', 'email': 'johndoe@example.com', 'signupDate': '2024-01-01T00:00:00Z'}}


In [22]:
messages = []
add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be very short.")
add_assistant_message(messages, "**\n```")
print(messages)
chat(messages, stop_sequences=["\n````"])

[{'role': 'user', 'content': 'Generate three different sample AWS CLI commands. Each should be very short.'}, {'role': 'assistant', 'content': '**\n```'}]


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011Cb57rjWBiHzwdyGtVSuAC'}